# CODICE PULITO di visualizzazione_dati_test
1. modifiche file singoli file merge (es. conversioni dummy, round age)
2. merge outer
3. creazione colonna globale per variabile comune ai due dataset
4. studio e applicazione strategia dove ci sono le differenze
5. ricordati ritrasformare dummy e cancellare x e y

appuntino --> age precedenza PTDEMOG


In [1]:
import pandas as pd

In [2]:
# --- 1. Caricamento dei dataset puliti ---
adnimerge = pd.read_csv("ADNIMERGE_cleaned_02.csv", parse_dates=["EXAMDATE", "update_stamp"])
ptdemog = pd.read_csv("PTDEMOG_cleaned_02.csv", parse_dates=["EXAMDATE", "update_stamp", "PTDOB"])

## Riconversione dummys

In [22]:
cols_x = sorted([c for c in ptdemog.columns if c.endswith('_0')])
coppie = []
for c in cols_x:
    base = c[:-2]
    c_y = base + '_1'
    if c_y in ptdemog.columns:
        coppie.append(base)

print("Coppie trovate:", coppie)

Coppie trovate: ['ETHNICITY', 'MARRY', 'RACE']


In [ ]:
# --- ADNIMERGE ---
for prefix, categories in [
    ("MARRY", [0, 1, 2, 3]),
    ("ETHNICITY", [0, 1]),
    ("RACE", [0, 1, 2, 3, 4, 5]),
]:
    cols_dummy = [f"{prefix}_{cat}" for cat in categories]
    ricostruita = pd.from_dummies(adnimerge[cols_dummy], sep="_", default_category="missing")
    adnimerge[prefix] = ricostruita[prefix]
    adnimerge = adnimerge.drop(columns=cols_dummy)

In [26]:
# --- PTDEMOG ---
for prefix, categories in [
    ("MARRY", [0, 1, 2, 3]),
    ("ETHNICITY", [0, 1]),
    ("RACE", [0, 1, 2, 3, 4, 5]),
]:
    cols_dummy = [f"{prefix}_{cat}" for cat in categories]
    ricostruita = pd.from_dummies(ptdemog[cols_dummy], sep="_", default_category="missing")
    ptdemog[prefix] = ricostruita[prefix]
    ptdemog = ptdemog.drop(columns=cols_dummy)

## Merge

In [28]:
keys = ["RID", "EXAMDATE"]

# Conto delle combinazioni uniche di chiavi
left_keys = adnimerge[keys].drop_duplicates()
right_keys = ptdemog[keys].drop_duplicates()

key_match = left_keys.merge(
    right_keys,
    on=keys,
    how="outer",
    indicator=True
)

counts = key_match["_merge"].value_counts()
print("Conteggio chiavi uniche per [RID, EXAMDATE]:")
print(counts)

print(f"Match: {counts.get('both', 0)}")
print(f"Solo in adnimerge: {counts.get('left_only', 0)}")
print(f"Solo in ptdemog: {counts.get('right_only', 0)}")

Conteggio chiavi uniche per [RID, EXAMDATE]:
_merge
left_only     8861
right_only    5515
both           445
Name: count, dtype: int64
Match: 445
Solo in adnimerge: 8861
Solo in ptdemog: 5515


In [29]:
# --- 2. Merge su RID + EXAMDATE ---
merged = pd.merge(
    adnimerge,
    ptdemog,
    on=keys,
    how="outer",
    indicator=True
)

In [30]:
# --- 3. Log di controllo post-merge ---
print(merged["_merge"].value_counts())
#merged = merged.drop(columns="_merge") #tenere per le visualizazioni eliminare solo alla fine

_merge
left_only     8861
right_only    5515
both           445
Name: count, dtype: int64


In [31]:
merged = merged[sorted(merged.columns)]
merged

,ADAS11,ADAS13,AGE_bl,AGE_x,AGE_y,APOE4,CDRSB,COLPROT,DX,DX_0,...,RAVLT_immediate,RID,VISCODE_x,VISCODE_y,VISIT_MONTH_x,VISIT_MONTH_y,Ventricles,_merge,update_stamp_x,update_stamp_y
0,NaN,NaN,NaN,NaN,60.711841,NaN,NaN,NaN,NaN,NaN,...,NaN,1,NaN,f,NaN,0.0,NaN,right_only,NaT,2005-08-18 00:00:00
1,NaN,NaN,NaN,NaN,74.379192,NaN,NaN,NaN,NaN,NaN,...,NaN,2,NaN,sc,NaN,0.0,NaN,right_only,NaT,2005-08-17 00:00:00
2,10.67,18.67,74.3,74.3,NaN,0.0,0.0,ADNI1,NaN,1.0,...,44.0,2,bl,NaN,0.0,NaN,118233.0,left_only,2023-07-07 04:59:40,NaT
3,NaN,NaN,NaN,NaN,79.477070,NaN,NaN,NaN,NaN,NaN,...,NaN,2,NaN,sc,NaN,61.0,NaN,right_only,NaT,2013-03-22 15:23:58
4,NaN,NaN,NaN,NaN,80.468172,NaN,NaN,NaN,NaN,NaN,...,NaN,2,NaN,m72,NaN,73.0,NaN,right_only,NaT,2013-05-30 10:05:05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14816,NaN,NaN,NaN,NaN,74.340862,NaN,NaN,NaN,NaN,NaN,...,NaN,10891,NaN,sc,NaN,0.0,NaN,right_only,NaT,2026-02-26 00:04:48
14817,NaN,NaN,NaN,NaN,76.585900,NaN,NaN,NaN,NaN,NaN,...,NaN,10893,NaN,sc,NaN,0.0,NaN,right_only,NaT,2026-02-26 00:04:48
14818,NaN,NaN,NaN,NaN,79.600274,NaN,NaN,NaN,NaN,NaN,...,NaN,10894,NaN,sc,NaN,0.0,NaN,right_only,NaT,2026-02-26 00:04:48
14819,NaN,NaN,NaN,NaN,67.356605,NaN,NaN,NaN,NaN,NaN,...,NaN,10895,NaN,sc,NaN,0.0,NaN,right_only,NaT,2026-02-26 00:04:48


## Creazione colonna unificata

In [32]:
both_rows = merged[sorted(merged.columns)]
both_rows[both_rows["_merge"] == "both"]

,ADAS11,ADAS13,AGE_bl,AGE_x,AGE_y,APOE4,CDRSB,COLPROT,DX,DX_0,...,RAVLT_immediate,RID,VISCODE_x,VISCODE_y,VISIT_MONTH_x,VISIT_MONTH_y,Ventricles,_merge,update_stamp_x,update_stamp_y
77,2.0,4.0,72.6,77.552772,77.678303,0.0,0.0,ADNIGO,NaN,1.0,...,58.0,21,m60,sc,59.0,60.0,18783.0,both,2023-07-07 04:59:41,2014-07-10 19:03:08
78,3.0,5.0,72.6,78.560301,78.685832,0.0,0.0,ADNI2,NaN,1.0,...,53.0,21,m72,m72,72.0,72.0,22013.0,both,2023-07-07 04:59:41,2013-05-30 10:05:05
96,5.0,10.0,71.7,76.817043,76.969199,0.0,0.0,ADNIGO,NaN,1.0,...,42.0,23,m60,sc,61.0,62.0,28003.0,both,2023-07-07 04:59:41,2013-03-22 15:23:58
97,6.0,10.0,71.7,77.813621,77.965777,0.0,0.0,ADNI2,NaN,1.0,...,39.0,23,m72,m72,73.0,74.0,29028.0,both,2023-07-07 04:59:41,2013-05-30 10:05:05
124,5.0,10.0,77.7,82.803354,82.915811,0.0,0.0,ADNIGO,NaN,1.0,...,53.0,31,m60,sc,61.0,62.0,31653.0,both,2023-07-07 04:59:41,2014-01-15 19:03:02
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6833,8.0,12.0,70.2,71.182888,71.321013,0.0,1.0,ADNI2,NaN,0.0,...,32.0,2396,m12,m12,12.0,13.0,49461.0,both,2023-07-07 04:59:53,2013-05-30 10:05:05
6842,14.0,26.0,71.5,72.529432,72.646133,2.0,4.0,ADNI2,NaN,0.0,...,28.0,2398,m12,m12,12.0,13.0,59320.0,both,2023-07-07 04:59:53,2013-05-30 10:05:05
6851,14.0,24.0,79.1,80.118480,80.298426,0.0,4.5,ADNI2,NaN,0.0,...,26.0,2403,m12,m12,12.0,14.0,39370.0,both,2023-07-07 04:59:53,2013-05-30 10:05:05
6859,10.0,16.0,71.6,72.596578,72.687201,0.0,0.5,ADNI2,NaN,0.0,...,47.0,2405,m12,m12,12.0,13.0,24946.0,both,2023-07-07 04:59:53,2013-05-30 10:05:05


In [34]:
# --- 2. Confronta ogni coppia, con tolleranza SOLO per le colonne numeriche ---
for base in coppie:
    col_x, col_y = f"{base}_x", f"{base}_y"

    if pd.api.types.is_numeric_dtype(both_rows[col_x]) and pd.api.types.is_numeric_dtype(both_rows[col_y]):
        diff_mask = (both_rows[col_x] - both_rows[col_y]).abs() > 0.01   # tolleranza per float
    else:
        diff_mask = both_rows[col_x] != both_rows[col_y]                # confronto esatto per stringhe/categorie

    n_diff = diff_mask.sum()
    print(f"{base}: {n_diff} righe diverse su {len(both_rows)}")

    if n_diff > 0:
        print(both_rows.loc[diff_mask, ['RID', col_x, col_y]].head(10))
        print()

ETHNICITY: 14378 righe diverse su 14821
   RID ETHNICITY_x ETHNICITY_y
0    1         NaN     missing
1    2         NaN           0
2    2           0         NaN
3    2         NaN           0
4    2         NaN           0
5    3         NaN           0
6    3           0         NaN
7    3           0         NaN
8    3           0         NaN
9    3           0         NaN

MARRY: 14397 righe diverse su 14821
   RID MARRY_x MARRY_y
0    1     NaN       1
1    2     NaN       1
2    2       1     NaN
3    2     NaN       3
4    2     NaN       3
5    3     NaN       1
6    3       1     NaN
7    3       1     NaN
8    3       1     NaN
9    3       1     NaN

RACE: 14378 righe diverse su 14821
   RID RACE_x   RACE_y
0    1    NaN  missing
1    2    NaN        5
2    2      5      NaN
3    2    NaN        5
4    2    NaN        5
5    3    NaN        5
6    3      5      NaN
7    3      5      NaN
8    3      5      NaN
9    3      5      NaN



In [35]:
# --- Blocco 1: controllo conflitti, solo sulle righe 'both' ---
both_mask = merged['_merge'] == 'both'

report_conflitti = {}
for base in coppie:
    col_x, col_y = f"{base}_x", f"{base}_y"
    entrambi_presenti = merged[col_x].notna() & merged[col_y].notna() & both_mask

    if pd.api.types.is_numeric_dtype(merged[col_x]) and pd.api.types.is_numeric_dtype(merged[col_y]):
        conflitto = entrambi_presenti & ((merged[col_x] - merged[col_y]).abs() > 0.01)
    else:
        conflitto = entrambi_presenti & (merged[col_x] != merged[col_y])

    report_conflitti[base] = conflitto.sum()
    print(f"{base}: {conflitto.sum()} conflitti su righe 'both'")

ETHNICITY: 2 conflitti su righe 'both'
MARRY: 21 conflitti su righe 'both'
RACE: 2 conflitti su righe 'both'


In [51]:
# --- Visualizza EXAMDATE, RID, update_stamp per le righe "both" ---
display_cols = ['RID', 'EXAMDATE', 'update_stamp_x', 'update_stamp_y', 'AGE_y', 'AGE_x', '_merge']
merged[merged["_merge"] == "both"][display_cols].head(10)

,RID,EXAMDATE,update_stamp_x,update_stamp_y,AGE_y,AGE_x,_merge
77,21,2010-10-07,2023-07-07 04:59:41,2014-07-10 19:03:08,77.678303,77.552772,both
78,21,2011-10-10,2023-07-07 04:59:41,2013-05-30 10:05:05,78.685832,78.560301,both
96,23,2010-12-21,2023-07-07 04:59:41,2013-03-22 15:23:58,76.969199,76.817043,both
97,23,2011-12-20,2023-07-07 04:59:41,2013-05-30 10:05:05,77.965777,77.813621,both
124,31,2010-12-01,2023-07-07 04:59:41,2014-01-15 19:03:02,82.915811,82.803354,both
125,31,2011-10-20,2023-07-07 04:59:41,2014-01-15 19:19:47,83.800137,83.687680,both
173,42,2011-04-13,2023-07-07 04:59:41,2013-11-22 19:19:32,78.362765,78.220945,both
234,55,2012-05-01,2023-07-07 04:59:41,2013-05-30 10:05:05,82.083504,81.797057,both
240,56,2010-12-10,2023-07-07 04:59:41,2013-03-22 15:23:58,74.776181,74.566461,both
241,56,2011-12-16,2023-07-07 04:59:41,2013-05-30 10:05:05,75.791923,75.582204,both


In [36]:
both_rows.dtypes

ADAS11                    float64
ADAS13                    float64
AGE_bl                    float64
AGE_x                     float64
AGE_y                     float64
APOE4                     float64
CDRSB                     float64
COLPROT                       str
DX                        float64
DX_0                      float64
DX_1                      float64
DX_2                      float64
EDUCATION_x               float64
EDUCATION_y               float64
ETHNICITY_x                   str
ETHNICITY_y                   str
EXAMDATE           datetime64[us]
Entorhinal                float64
FAQ                       float64
FSVERSION                 float64
Fusiform                  float64
GENDER_x                      str
GENDER_y                      str
HAS_QC_ERROR              float64
Hippocampus               float64
ICV                       float64
IMAGEUID                  float64
MARRY_x                       str
MARRY_y                       str
MMSE          

In [65]:
# --- Controllo conflitti e gestione AGE ---
both = merged['_merge'].eq('both')
report_conflitti = {}

# Tutte le variabili che hanno sia _x che _y
basi = sorted(
    c[:-2] for c in merged.columns if c.endswith('_x')
    and f"{c[:-2]}_y" in merged.columns
)

for base in basi:
    x, y = f"{base}_x", f"{base}_y"
    presenti = both & merged[x].notna() & merged[y].notna()

    if pd.api.types.is_numeric_dtype(merged[x]):
        conflitto = presenti & merged[x].sub(merged[y]).abs().gt(0.01)
    else:
        conflitto = presenti & merged[x].ne(merged[y])

    n = conflitto.sum()
    report_conflitti[base] = n

    print(f"{base}: {n} conflitti su righe 'both'")

    if n:
        print(merged.loc[conflitto, ['RID', x, y]].head(10), '\n')

# AGE: precedenza a ptdemog (AGE_y)
if 'AGE_y' in merged.columns:
    merged['AGE'] = merged['AGE_y'].combine_first(merged['AGE_x'])
    

AGE: 445 conflitti su righe 'both'
     RID      AGE_x      AGE_y
77    21  77.552772  77.678303
78    21  78.560301  78.685832
96    23  76.817043  76.969199
97    23  77.813621  77.965777
124   31  82.803354  82.915811
125   31  83.687680  83.800137
173   42  78.220945  78.362765
234   55  81.797057  82.083504
240   56  74.566461  74.776181
241   56  75.582204  75.791923 

EDUCATION: 4 conflitti su righe 'both'
       RID  EDUCATION_x  EDUCATION_y
1631   376         15.0         14.0
2645   625          6.0          8.0
6011  2079         13.0         12.0
6371  2210         12.0         13.0 

ETHNICITY: 2 conflitti su righe 'both'
       RID ETHNICITY_x ETHNICITY_y
5856  2022     missing           0
6175  2146           1           0 

GENDER: 0 conflitti su righe 'both'
MARRY: 21 conflitti su righe 'both'
      RID MARRY_x  MARRY_y
96     23       3        1
97     23       3        1
982   214       1        3
1359  303       1        3
1390  311       1        3
1663  382       

In [66]:
# --- Blocco 2: coalesce, SENZA eliminare _x/_y ---

for base in basi:
    x, y = f"{base}_x", f"{base}_y"

    # AGE: priorità a ptdemog (y)
    # Altre variabili: priorità a _x
    merged[base] = (
        merged[y].combine_first(merged[x])
        if base == "AGE"
        else merged[x].combine_first(merged[y])
    )

In [70]:
display_cols = ['RID', 'VISCODE', 'VISIT_MONTH', 'ETHNICITY', 'RACE', 'EDUCATION', '_merge']
merged[merged["_merge"] == "both"][display_cols].head(10)

,RID,VISCODE,VISIT_MONTH,ETHNICITY,RACE,EDUCATION,_merge
77,21,m60,59.0,0,4,18.0,both
78,21,m72,72.0,0,4,18.0,both
96,23,m60,61.0,0,4,14.0,both
97,23,m72,73.0,0,4,14.0,both
124,31,m60,61.0,0,5,18.0,both
125,31,m72,72.0,0,5,18.0,both
173,42,m60,65.0,0,5,18.0,both
234,55,m72,76.0,0,5,20.0,both
240,56,m60,60.0,0,4,13.0,both
241,56,m72,72.0,0,4,13.0,both


In [69]:
merged

,ADAS11,ADAS13,AGE_bl,AGE_x,AGE_y,APOE4,CDRSB,COLPROT,DX,DX_0,...,update_stamp_y,ETHNICITY,MARRY,RACE,AGE,EDUCATION,GENDER,VISCODE,VISIT_MONTH,update_stamp
0,NaN,NaN,NaN,NaN,60.711841,NaN,NaN,NaN,NaN,NaN,...,2005-08-18 00:00:00,missing,1,missing,60.711841,18.0,0,f,0.0,2005-08-18 00:00:00
1,NaN,NaN,NaN,NaN,74.379192,NaN,NaN,NaN,NaN,NaN,...,2005-08-17 00:00:00,0,1,5,74.379192,16.0,1,sc,0.0,2005-08-17 00:00:00
2,10.67,18.67,74.3,74.3,NaN,0.0,0.0,ADNI1,NaN,1.0,...,NaT,0,1,5,74.300000,16.0,1,bl,0.0,2023-07-07 04:59:40
3,NaN,NaN,NaN,NaN,79.477070,NaN,NaN,NaN,NaN,NaN,...,2013-03-22 15:23:58,0,3,5,79.477070,16.0,1,sc,61.0,2013-03-22 15:23:58
4,NaN,NaN,NaN,NaN,80.468172,NaN,NaN,NaN,NaN,NaN,...,2013-05-30 10:05:05,0,3,5,80.468172,16.0,1,m72,73.0,2013-05-30 10:05:05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14816,NaN,NaN,NaN,NaN,74.340862,NaN,NaN,NaN,NaN,NaN,...,2026-02-26 00:04:48,0,1,5,74.340862,20.0,1,sc,0.0,2026-02-26 00:04:48
14817,NaN,NaN,NaN,NaN,76.585900,NaN,NaN,NaN,NaN,NaN,...,2026-02-26 00:04:48,0,3,5,76.585900,12.0,0,sc,0.0,2026-02-26 00:04:48
14818,NaN,NaN,NaN,NaN,79.600274,NaN,NaN,NaN,NaN,NaN,...,2026-02-26 00:04:48,0,3,5,79.600274,14.0,1,sc,0.0,2026-02-26 00:04:48
14819,NaN,NaN,NaN,NaN,67.356605,NaN,NaN,NaN,NaN,NaN,...,2026-02-26 00:04:48,0,1,5,67.356605,16.0,0,sc,0.0,2026-02-26 00:04:48
